In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
import my_data_manager as mdm

cat = mdm.load_cat("Data/galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)
clusters = clusters[:678]


/home/gasep/Projects/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/home/gasep/Projects/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [3]:
clean_clusters = [df[df.groupby("haloId")["haloId"].transform("count") > 3].reset_index(drop=True) for df in clusters]

In [4]:
import pandas as pd

N = 21

splus_clusters = [df[pd.to_numeric(df['mag_r'], errors = 'coerce') >= N].copy() for df in clusters]

In [5]:
cluster_samples = {
    #"Raw": clusters,
    "Clean": clean_clusters,
    "SPLUS": splus_clusters
}

## Clustering

In [6]:
import clustering_methods as clustering
import numpy as np

algorithms = {}


algorithms['BGMM'] = clustering.run_GMM
algorithms['DBSCAN'] = clustering.run_DBSCAN
algorithms['HDBSCAN'] = clustering.run_HDBSCAN
algorithms['Optics'] = clustering.run_OPTICS
algorithms['Kmeans'] = clustering.run_Kmeans
algorithms['Agglomerative'] = clustering.run_Aglomerative_Clustering
algorithms['Affinity'] = clustering.run_Affinity_Propagation

params = {
    "max_clusters" : 0.1,
    "covariance_type" : 'full',
    "min_cluster_size" : 4,
    "min_eps" : 0.5,
    "max_eps" : np.inf,
    "linkage" : 'ward',
    "clustering_threshold" : 0.5,
    "max_iter" : 1000
}

In [7]:
import time

def ml_worker(alg, samples):

    predictions = []
    # 1. Initialize as a list instead of a scalar
    iteration_times = [] 

    for sample in samples:
    
        X_data = sample["RA"].values
        Y_data = sample["DEC"].values
        
        data = np.column_stack((X_data, Y_data))
        data = np.asarray(data, dtype=np.float64)

        start_time = time.perf_counter()
        labels, probs, c = algorithms[alg](data, params)
        end_time = time.perf_counter()
        
        delta_time = end_time - start_time
        # 2. Store the specific time for this iteration
        iteration_times.append(delta_time) 
        
        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
        )

        fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
        labels = np.insert(labels.astype('<U32'), 0, str(fof_id))

        predictions.append(labels)
    
    return iteration_times, predictions

In [6]:
from ds_plus import milaDS
import astro_utils as au
import numpy as np
import time
from astropy.stats import biweight_location


def dsp_worker(samples):

    predictions = []
    # 1. Change from scalar to list
    iteration_times = []

    for sample in samples:
        
        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)
        
        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)

        X_Kpc, Y_Kpc = au.gal_Mpc_coords(X_data, Y_data, Z_data, x_cluster, y_cluster)
        X_Kpc *= 1000
        Y_Kpc *= 1000
        
        V_data = au.los_vel(Z_data, Z_clus)

        # --- Timing Start ---
        start_time = time.perf_counter()
        galaxy_info, grouping, summary = milaDS.DSp_groups(X_data, Y_data, V_data, Z_clus)
        end_time = time.perf_counter()
        
        delta_time = end_time - start_time
        # 2. Store individual iteration time
        iteration_times.append(delta_time)
        # --- Timing End ---
        
        # Extract the 9th column (group ID)
        labels = np.array([row[8] for row in grouping])

        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
        )

        fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
        # Insert the fof_id at the start of the array
        labels = np.insert(labels.astype(str), 0, fof_id)

        predictions.append(labels)

    # 3. Return the list of times
    return iteration_times, predictions

In [9]:
from calsagos import lagasu
from calsagos import utils
from calsagos import clumberi
from astropy.stats import biweight_location
import numpy as np
import time

from IPython.display import clear_output

def calsagos_worker(samples):
    #- S-PLUS mock cosmology
    H_mock = 67.3
    Omega_L_mock = 0.685
    Omega_m_mock = 0.315

    range_cut_percentage = 0.2
    #-- GENERAL PARAMETERS
    n_galaxies = 4 # -- number of minimum of galaxies that a group or substructure must have

    predictions = []
    total_time = 0

    for sample in samples:

        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)

        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)
        
        cluster_mass = float(sample["log(m_200)"].iloc[0])
        
        range_cuts = int(len(sample)*range_cut_percentage)
        
        start_time = time.perf_counter()
        id_galaxy = indexes = np.arange(len(sample) + 1)
        
        #-- defining the cluster radius
        r200_kpc = utils.calc_radius_finn(cluster_mass, Z_clus, H_mock, Omega_L_mock, Omega_m_mock, "kiloparsec")

        #-- converting the radius in kpc to a radius in angular units
        #-- NOTE: if the user has an estimate of the r200 of the cluster it is not necessary to calculate this quantity
        r200_degree = utils.convert_kpc_to_angular_distance(r200_kpc, Z_clus, H_mock, Omega_m_mock, "degrees") 

        #-- select cluster members
        cluster_members = clumberi.clumberi(id_galaxy, X_data, Y_data, Z_data, Z_clus, x_cluster, y_cluster, range_cuts)

        # -- defining output parameters from clumberi
        id_member = cluster_members[0]
        ra_member = cluster_members[1]
        dec_member = cluster_members[2]
        redshift_member = cluster_members[3]


        #-- estimating the galaxy separation of galaxies in the cluster sample to be used as input in lagasu
        knn_distance = utils.calc_knn_galaxy_distance(ra_member, dec_member, n_galaxies)

        #-- determining the distance to the k-nearest neighbor of each galaxy in the cluster
        knn_galaxy_distance = knn_distance[0]

        try:
            typical_separation = utils.best_eps_dbscan(id_member, knn_galaxy_distance)
        except:
            clear_output(wait=True)
            print("Error ")
            print(id_member)
            print(len(id_member))
            print(knn_distance)
            print(len(knn_distance))

        #-- Assign galaxies to each substructures
        label_candidates = lagasu.lagasu(id_galaxy, X_data, Y_data, Z_data, 
                            range_cuts, typical_separation, n_galaxies, 'euclidean', 'dbscan', 
                            x_cluster, y_cluster, Z_clus, 
                            r200_degree, 'zspec')
        
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time

        #-- defining output parameters from lagasu
        id_candidates = label_candidates[0]
        ra_candidates = label_candidates[1]
        dec_candidates = label_candidates[2]
        redshift_candidates = label_candidates[3]
        label_zcut = label_candidates[4]
        label_final = label_candidates[5]    
    
        
        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
        )

        fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
        labels = np.insert(label_final.astype(str), 0, fof_id)

        predictions.append(label_final)

    return total_time, predictions

In [10]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_ml_worker(worker, alg, thread_count, iterations, samples):
    results = []
    
    with ProcessPoolExecutor(max_workers=thread_count) as executor:
            futures = [executor.submit(worker, alg, samples) for _ in range(iterations)]
            for future in as_completed(futures):
                results.append(future.result())
    
    return results

In [7]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_worker(worker, thread_count, iterations, samples):
    results = []

    with ProcessPoolExecutor(max_workers=thread_count) as executor:
        futures = [executor.submit(worker, samples) for _ in range(iterations)]

        for future in as_completed(futures):
            results.append(future.result())
    
    return results

In [8]:
from openpyxl import Workbook
from itertools import zip_longest

def save_xlsx(results, title):
    wb = Workbook()

    # 1. Setup the Execution Times sheet (Rows = Iterations, Cols = Samples/Runs)
    sheet1 = wb.active
    sheet1.title = "Execution Times"
    
    # Extract just the duration lists from the results
    # results = [( [times], [preds] ), ( [times], [preds] )]
    all_duration_lists = [res[0] for res in results]
    
    # Create Headers: "Iteration", "Sample 1", "Sample 2", etc.
    headers = ["Iteration"] + [f"Run {i+1}" for i in range(len(all_duration_lists))]
    sheet1.append(headers)

    # Use zip_longest to pair up times by iteration index
    # fillvalue="" handles cases where one run has fewer iterations than others
    for idx, row_times in enumerate(zip_longest(*all_duration_lists, fillvalue="")):
        # Append iteration number (idx+1) followed by the times for that iteration
        sheet1.append([idx + 1] + list(row_times))

    # 2. Store the prediction data in separate sheets as before
    for res_idx, (_, data) in enumerate(results):
        sheet = wb.create_sheet(title=f"Result_{res_idx + 1}")
        
        # This keeps your original logic for predictions
        for row in zip_longest(*data, fillvalue=""):
            sheet.append(row)

    wb.save(f"{title}.xlsx")

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
threads = 4
iterations = 10

for cs in cluster_samples.keys():
    cluster_sample = splus_clusters

    folder = f"Results/{cs}_samples"
    os.makedirs(folder, exist_ok=True)

    #for alg in algorithms.keys():
        #ml_results = run_ml_worker(ml_worker, alg, threads, iterations, cluster_sample)
        #save_xlsx(ml_results, os.path.join(folder,f"ML_{alg}"))
        #print(f"saved to {folder}/{alg}")

    dsp_results = run_worker(dsp_worker, threads, iterations, cluster_sample)
    save_xlsx(dsp_results, os.path.join(folder,f"DSP"))
    print(f"saved to {folder}/DSP")

    #calsagos_results = run_worker(calsagos_worker, threads, iterations, cluster_sample)
    #save_xlsx(calsagos_results, os.path.join(folder,f"CALSAGOS"))

saved to Results/Clean_samples/DSP


## Results Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Visual Configurations ---
thresholds = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
indices = [0, 20, 40, 60, 80, 99]
layer_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

all_possible_methods = sorted([
    "CALSAGOS", "DSP", "ML_DBSCAN", "ML_HDBSCAN", "ML_Optics",
    "ML_Kmeans", "ML_Affinity", "ML_GMM", "ML_Aglomerative"
]) 

metric_trios = [
    ("Neighborhood", ["Neighborhood_Completeness", "Neighborhood_Purity", "Neighborhood_F1_Score"]),
    ("Membership", ["Membership_Completeness", "Membership_Purity", "Membership_F1_Score"])
]

scenarios = ["all", "no_noise", "satellites", "center"]
titles = ["All Data", "No Single Galaxies", "Satellites Only", "Center Only"]

def plot_horizontal_trio_large_legend(key, trio_name, trio_list):
    for s_idx, mode in enumerate(scenarios):
        fig, axes = plt.subplots(1, 3, figsize=(26, 10), sharey=True)
        valid_trio = False
        
        for i, metric in enumerate(trio_list):
            file_path = f"{key}_{metric}_{mode}_AVG.xlsx"
            ax = axes[i]
            
            if not os.path.exists(file_path):
                ax.text(0.5, 0.5, f"Data Missing:\n{file_path}", ha='center', va='center')
                continue
            
            valid_trio = True
            df_avg = pd.read_excel(file_path, index_col=0)
            methods_present = [m for m in all_possible_methods if m in df_avg.index]
            df_plot = df_avg.loc[methods_present]
            
            x = np.arange(len(methods_present))
            
            for j, idx in enumerate(indices):
                vals = df_plot.iloc[:, idx].values
                lbl = "$t = 1.0$" if thresholds[j] == 1.0 else f"$t > {thresholds[j]:.1f}$"
                current_label = lbl if i == 0 else ""

                ax.bar(x, vals, width=0.75, color=layer_colors[j], label=current_label,
                       edgecolor='white', linewidth=0.7, zorder=j+2) # Higher zorder to stay above grid
                
                for k, val in enumerate(vals):
                    if val > 12: 
                        ax.text(k, val - 1.8, f'{val:.0f}%', ha='center', va='top', 
                                fontsize=9, fontweight='bold', color='white', zorder=j+3)

            # --- Updated Tick and Grid Polish ---
            metric_label = " ".join(metric.split('_')[1:])
            ax.set_title(metric_label, fontsize=28, fontweight='bold', pad=15)
            
            # Set Y-ticks every 5 units
            ax.set_ylim(0, 115)
            ax.set_yticks(np.arange(0, 120, 5)) 
            
            # Enable Y-axis grid
            ax.grid(axis='y', linestyle=':', color='gray', alpha=0.4, linewidth=0.8, zorder=0)
            
            ax.set_xticks(x)
            ax.set_xticklabels(methods_present, rotation=35, ha='right', fontsize=12)
            
            if i == 0:
                ax.set_ylabel("Absolute Success (%)", fontsize=16, fontweight='bold')
            
            for s in ['top', 'right']: ax.spines[s].set_visible(False)

        if valid_trio:
            fig.suptitle(f"{key.upper()} {trio_name}: {titles[s_idx]}", 
                         fontsize=26, fontweight='bold', y=1.12)
            
            handles, labels = axes[0].get_legend_handles_labels()
            fig.legend(handles, labels, loc='upper center', 
                       bbox_to_anchor=(0.5, 1.06), 
                       ncol=6, 
                       title="Reliability Thresholds", 
                       frameon=False, 
                       fontsize=22,          
                       title_fontsize=20,     
                       columnspacing=1.5,     
                       handletextpad=0.5)     
            
            plt.tight_layout()
            out_file = f"{key}_{trio_name}_{mode}_horizontal.png"
            plt.savefig(out_file, dpi=300, bbox_inches='tight')
            plt.show()
        plt.close(fig)

# --- Execution ---
keys = ["raw", "clean", "splus"]
for k in keys:
    for name, m_list in metric_trios:
        plot_horizontal_trio_large_legend(k, name, m_list)

In [ ]:
import numpy as np
import pickle
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_metric_distance_correlation(metric_pkl, distance_pkl, method="ML_HDBSCAN"):
    with open(metric_pkl, "rb") as f:
        metric_data = pickle.load(f)[method]
    with open(distance_pkl, "rb") as f:
        dist_data = pickle.load(f)[method]
        
    all_metrics = []
    all_distances = []
    
    for h_id in metric_data.keys():
        # Flatten matrices
        m_vals = metric_data[h_id].flatten()
        d_vals = dist_data[h_id].flatten()
        
        # We only care about pairs that have SOME overlap (Metric > 0)
        # And valid distances (Distance >= 0)
        mask = (m_vals > 0) & (d_vals >= 0)
        
        all_metrics.extend(m_vals[mask])
        all_distances.extend(d_vals[mask])
        
    # Calculate Correlation
    corr, p_value = spearmanr(all_distances, all_metrics)
    
    return np.array(all_distances), np.array(all_metrics), corr, p_value

# --- Execute Analysis ---
# Example: Using Membership Completeness and Centroid Distance
dist_vals, comp_vals, rho, p = analyze_metric_distance_correlation(
    "raw_Membership_Completeness_results.pkl", 
    "raw_Centroid_Distances_results.pkl"
)

print(f"Spearman Correlation (Rho): {rho:.3f}")
print(f"P-value: {p:.3e}")

plt.figure(figsize=(10, 8))
plt.hexbin(dist_vals, comp_vals, gridsize=30, cmap='viridis', bins='log')
plt.colorbar(label='Log10(Count of True-Pred Pairs)')
plt.title(f"Distance vs. Completeness Correlation\n(Spearman Rho: {rho:.3f})", fontsize=16)
plt.xlabel("Distance (Mpc or arcmin)", fontsize=12)
plt.ylabel("Completeness Fraction", fontsize=12)
plt.show()